# 04 — Systematic Prompt Testing Pipeline

## Overview

A systematic **4-phase pipeline** to identify the optimal LLM configuration for
screening papers in Cochrane systematic reviews. Each phase builds on the previous,
progressively refining the approach.

| Phase | Question | Data | Models |
|-------|----------|------|--------|
| **1. Prompt Optimisation** | Best prompt per context type? | 3 batches × 200 | all models | 
| **2. Few-Shot & Model Comparison** | Best examples? Best models? | Full dataset | all models | 
| **3. Full Evaluation + Reasoning** | Does CoT / reasoning help? | Full dataset | all models |
| **4. Multi-Agent** | Does multi-agent improve screening? | Full dataset | only council approach |

### Key Design Decisions
- **Primary metric:** F₂ score — weights recall 4× over precision (β²=4, so missing a relevant paper is penalised far more than including an irrelevant one)
- **Temperature:** 0.0 — deterministic, reproducible outputs for binary classification
- **Context variants tested:** Ex-Post (full review abstract available) vs. Ex-Ante (objectives + selection criteria only)
- **Dual-winner design:** Phase 1 selects the best prompt for EACH context type, and both are carried through all subsequent phases **-> to be discussed, was agreed differently before (implementation currently without)**

### Models
- **Local (Ollama):** Llama3.1 8b, Mixtral 8x7b, Gemma3 27b, Mistral Small3.2 24b
- **Medical:** Meditron 7b
- **Proprietary:** GPT-4o-mini
- **Reasoning:** Gemini 2.5 Flash

---

## 1. Setup

Import required libraries, configure API clients, and define pipeline constants. This section initializes Ollama (local models), OpenAI, and Google GenAI clients, and registers all models to be evaluated.

In [ ]:
# Import required libraries for multi-phase LLM evaluation pipeline.
# Core dependencies: pandas/numpy for data, sklearn for metrics, tqdm for progress,
# ollama/openai/google for LLM inference, statsmodels for inter-rater statistics.

# ─────────────────────────────────────────────────────────────────────────────
# Standard library imports
# ─────────────────────────────────────────────────────────────────────────────
import os, re, time, json, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# ─────────────────────────────────────────────────────────────────────────────
# Data processing and machine learning
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, fbeta_score, confusion_matrix)
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

# ─────────────────────────────────────────────────────────────────────────────
# LLM API clients (local and proprietary)
# ─────────────────────────────────────────────────────────────────────────────
import ollama                                  # Local models via Ollama
from dotenv import load_dotenv                 # Environment variable loading
import openai                                  # OpenAI GPT models
from google import genai                       # Google Gemini models
from google.genai import types as genai_types  # Gemini configuration types

In [ ]:
# Configure file paths and load environment variables.
# Sets up project directories and creates results folder for storing evaluation outputs.

# ─────────────────────────────────────────────────────────────────────────────
# Determine project root (handles running from notebook dir or project root)
# ─────────────────────────────────────────────────────────────────────────────
notebook_dir = Path.cwd()
project_root = notebook_dir if (notebook_dir / 'Data').exists() else notebook_dir.parent

# Load API keys from .env file
load_dotenv(project_root / '.env')

# ─────────────────────────────────────────────────────────────────────────────
# Configure directories for input data and output results
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR     = project_root / 'Data'
RESULTS_DIR  = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)  # Create results directory if needed

In [ ]:
# Initialize API clients for proprietary LLM providers.
# API keys are loaded from environment variables (OPEN_AI_API_KEY, GEMINI_API_KEY).
# These clients are used for GPT-4o-mini and Gemini 2.5 Flash evaluations.

# ─────────────────────────────────────────────────────────────────────────────
# Create OpenAI client (for GPT models)
# ─────────────────────────────────────────────────────────────────────────────
openai_client = openai.OpenAI(api_key=os.environ.get('OPEN_AI_API_KEY'))

# ─────────────────────────────────────────────────────────────────────────────
# Create Google GenAI client (for Gemini models)
# ─────────────────────────────────────────────────────────────────────────────
google_client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

In [ ]:
# Define pipeline constants for reproducibility and sampling configuration.
# TEMPERATURE=0.0 ensures deterministic outputs for binary classification.

# ─────────────────────────────────────────────────────────────────────────────
# Model inference settings
# ─────────────────────────────────────────────────────────────────────────────
TEMPERATURE  = 0.0   # Deterministic output (no randomness)
RANDOM_STATE = 42    # Seed for reproducible sampling

# ─────────────────────────────────────────────────────────────────────────────
# Sampling configuration for Phase 1 prompt optimization
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_SIZE  = 200   # Papers per batch
N_BATCHES    = 3     # Number of evaluation batches
N_REVIEWS    = 8     # Reviews to sample from
N_INC_PER_REV = 5    # Included papers per review per batch
N_EXC_PER_REV = 20   # Excluded papers per review per batch

In [ ]:
# Define the model registry with all LLMs to be evaluated.
# Organized by category: local Ollama models, medical-domain models, and proprietary reasoning models.
# Each entry contains: (model_id, save_name, backend).

# ─────────────────────────────────────────────────────────────────────────────
# Local Ollama models (general-purpose LLMs)
# ─────────────────────────────────────────────────────────────────────────────
OLLAMA_MODELS = [
    ('llama3.1:8b',         'llama3_1_8b',          'ollama'),   # Meta Llama 3.1 8B
    ('mixtral:8x7b',        'mixtral_8x7b',         'ollama'),   # Mistral MoE 8x7B
    ('gemma3:27b',          'gemma3_27b',           'ollama'),   # Google Gemma 3 27B
    ('mistral-small3.2:24b','mistral_small3_2_24b', 'ollama'),   # Mistral Small 3.2 24B
]

# ─────────────────────────────────────────────────────────────────────────────
# Medical-domain models (specialized for biomedical text)
# ─────────────────────────────────────────────────────────────────────────────
MEDICAL_MODELS = [
    ('MedAIBase/MedGemma1.0:4b', 'medgemma_4b', 'ollama'),  # MedGemma biomedical LLM
]

# ─────────────────────────────────────────────────────────────────────────────
# Proprietary reasoning models (with API access)
# ─────────────────────────────────────────────────────────────────────────────
REASONING_MODELS = [
    ('gpt-4o-mini',      'gpt_4o_mini',      'openai'),   # OpenAI GPT-4o-mini
    ('gemini-2.5-flash', 'gemini_2_5_flash', 'google'),   # Google Gemini 2.5 Flash
]

# Combined list of all models for evaluation
ALL_MODELS = OLLAMA_MODELS + MEDICAL_MODELS + REASONING_MODELS

In [ ]:
# Verify all local Ollama models are installed and API clients are configured.
# Reports any missing models and confirms configuration status.

# ─────────────────────────────────────────────────────────────────────────────
# Check which Ollama models are installed locally
# ─────────────────────────────────────────────────────────────────────────────
available_ollama = [m.model for m in ollama.list().models]
all_local = OLLAMA_MODELS + MEDICAL_MODELS

# Warn about any missing local models
for model_id, name, backend in all_local:
    if not any(model_id in a for a in available_ollama):
        print(f"⚠ Missing Ollama model: {model_id}")

# ─────────────────────────────────────────────────────────────────────────────
# Print configuration summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"Ollama models installed: {len(available_ollama)}")
print(f"API clients ready: OpenAI ✓  Google ✓")
print(f"Temperature: {TEMPERATURE}  |  Random seed: {RANDOM_STATE}")

Ollama models installed: 26
API clients ready: OpenAI ✓  Google ✓
Temperature: 0.0  |  Random seed: 42


---

## 2. Data Preparation

Load the ground truth validation dataset from NB03. Extract structured sections (Objectives, Selection criteria) from Cochrane review abstracts. Filter to reviews with both included and excluded papers, then create stratified batches for prompt optimization.

In [ ]:
# Define regex patterns and extraction function for parsing Cochrane structured abstracts.
# Cochrane abstracts follow a standard format with labeled sections.

# ─────────────────────────────────────────────────────────────────────────────
# Regex pattern for detecting section breaks in Cochrane abstracts
# Matches section headers like "Background:", "Objectives:", "Main results:", etc.
# ─────────────────────────────────────────────────────────────────────────────
SECTION_BREAK_RE = r"\n\s*(?:background|objectives?|search methods|selection criteria|data collection and analysis|main results|authors' conclusions)\s*:"

# ─────────────────────────────────────────────────────────────────────────────
# Patterns for extracting specific sections
# ─────────────────────────────────────────────────────────────────────────────
OBJECTIVES_RE = re.compile(
    rf"(?is)(?:^|\n)\s*objectives?\s*:\s*(.+?)(?={SECTION_BREAK_RE}|$)"
)
SELECTION_RE = re.compile(
    rf"(?is)(?:^|\n)\s*selection\s*criteria\s*:\s*(.+?)(?={SECTION_BREAK_RE}|$)"
)


def extract_section_text(review_abstract: str, section_regex: re.Pattern) -> str:
    """
    Extract a named section body from a Cochrane structured abstract.
    
    Args:
        review_abstract: Full text of the Cochrane review abstract
        section_regex: Compiled regex pattern for the target section
        
    Returns:
        Extracted section text with normalized whitespace, or empty string if not found
    """
    if pd.isna(review_abstract):
        return ''
    match = section_regex.search(str(review_abstract))
    if not match:
        return ''
    # Normalize whitespace (collapse multiple spaces/newlines to single space)
    return re.sub(r"\s+", " ", match.group(1).strip())

In [ ]:
# Define path to the ground truth validation dataset created in notebook 03.

ground_truth_csv = DATA_DIR / 'ground_truth_validation_dataset.csv'

In [ ]:
# Load the ground truth validation dataset with review context and paper abstracts.
# Creates a stable review_id field for consistent identification across exports.

# ─────────────────────────────────────────────────────────────────────────────
# Load dataset with required columns for evaluation
# ─────────────────────────────────────────────────────────────────────────────
eval_data = pd.read_csv(
    ground_truth_csv,
    usecols=['review_doi', 'review_title', 'review_abstract', 'paper_title', 'paper_abstract', 'label'],
    low_memory=False,
)

# ─────────────────────────────────────────────────────────────────────────────
# Normalize review identifiers for consistent matching
# ─────────────────────────────────────────────────────────────────────────────
eval_data['review_id'] = eval_data['review_doi'].astype(str).str.upper().str.strip()

# Display sample and shape for verification
eval_data.head(3)
print(eval_data.shape)

(10225, 7)


In [ ]:
# Extract the Objectives and Selection Criteria sections from each review abstract.
# These fields are used for the ex-ante context variant in prompt evaluation,
# simulating a scenario where only the protocol (not results) is available.

# ─────────────────────────────────────────────────────────────────────────────
# Apply section extraction to all reviews
# ─────────────────────────────────────────────────────────────────────────────
eval_data['objectives_text'] = eval_data['review_abstract'].apply(
    lambda x: extract_section_text(x, OBJECTIVES_RE)
)
eval_data['selection_criteria_text'] = eval_data['review_abstract'].apply(
    lambda x: extract_section_text(x, SELECTION_RE)
)

In [ ]:
# Filter dataset to rows where both Objectives and Selection Criteria were successfully extracted.
# Papers without these sections cannot be evaluated using the ex-ante context variant.

# ─────────────────────────────────────────────────────────────────────────────
# Keep only rows with non-empty extracted sections
# ─────────────────────────────────────────────────────────────────────────────
phase_ready_data = eval_data[
    eval_data['objectives_text'].str.strip().ne('')
    & eval_data['selection_criteria_text'].str.strip().ne('')
].copy()

In [ ]:
# Filter to reviews that have both INCLUDE and EXCLUDE papers for valid evaluation.
# Ensures each review can contribute to both positive and negative class metrics.

# ─────────────────────────────────────────────────────────────────────────────
# Calculate per-review label counts
# ─────────────────────────────────────────────────────────────────────────────
review_labels = phase_ready_data.groupby('review_doi')['label'].agg(['sum', 'count']).reset_index()
review_labels.columns = ['review_doi', 'n_included', 'n_total']
review_labels['n_excluded'] = review_labels['n_total'] - review_labels['n_included']

# ─────────────────────────────────────────────────────────────────────────────
# Filter to reviews with at least 1 paper in each class
# ─────────────────────────────────────────────────────────────────────────────
reviews_with_both = review_labels[
    (review_labels['n_included'] > 0) & (review_labels['n_excluded'] > 0)
]

# Apply filter to main dataset
eval_data = phase_ready_data[
    phase_ready_data['review_doi'].isin(reviews_with_both['review_doi'])
].copy()

# Store labels for metrics computation
labels_full = eval_data['label'].tolist()

# ─────────────────────────────────────────────────────────────────────────────
# Print data quality summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"Rows loaded: {len(phase_ready_data):,}")
print(f"Eligible for evaluation: {len(eval_data):,} ({eval_data['review_doi'].nunique()} reviews)")
print(f"Rows missing objectives_text: {(eval_data['objectives_text'].str.strip() == '').sum()}")
print(f"Rows missing selection_criteria_text: {(eval_data['selection_criteria_text'].str.strip() == '').sum()}")

print(eval_data.shape)
print(eval_data.columns.tolist())

Rows loaded: 10,225
Eligible for evaluation: 10,225 (30 reviews)
Rows missing objectives_text: 0
Rows missing selection_criteria_text: 0
(10225, 9)
['review_doi', 'review_title', 'review_abstract', 'paper_title', 'paper_abstract', 'label', 'review_id', 'objectives_text', 'selection_criteria_text']


In [ ]:
# Create stratified batches for Phase 1 prompt optimization.
# Selects reviews with sufficient papers per class, then samples balanced batches
# with N_INC_PER_REV included and N_EXC_PER_REV excluded papers per review.

# ─────────────────────────────────────────────────────────────────────────────
# Calculate per-review paper counts by class
# ─────────────────────────────────────────────────────────────────────────────
review_counts = (
    eval_data.groupby('review_doi')['label']
    .agg(n_inc='sum', n_total='count')
    .assign(n_exc=lambda x: x['n_total'] - x['n_inc'])
    .reset_index()
)

# ─────────────────────────────────────────────────────────────────────────────
# Filter to reviews with enough papers of each class for balanced sampling
# ─────────────────────────────────────────────────────────────────────────────
eligible_reviews = review_counts[
    (review_counts['n_inc'] >= N_INC_PER_REV) &
    (review_counts['n_exc'] >= N_EXC_PER_REV)
]

print(f"Reviews with enough papers (>={N_INC_PER_REV} INC, >={N_EXC_PER_REV} EXC): {len(eligible_reviews)}")

# ─────────────────────────────────────────────────────────────────────────────
# Randomly select N_REVIEWS reviews for batch creation
# ─────────────────────────────────────────────────────────────────────────────
chosen_reviews = eligible_reviews['review_doi'].sample(
    n=N_REVIEWS, random_state=RANDOM_STATE
).tolist()

print(f"Chosen reviews: {len(chosen_reviews)}")

# ─────────────────────────────────────────────────────────────────────────────
# Create N_BATCHES stratified batches
# Each batch samples N_INC_PER_REV included and N_EXC_PER_REV excluded per review
# ─────────────────────────────────────────────────────────────────────────────
batches = []
for i in range(N_BATCHES):
    batch_parts = []
    seed = RANDOM_STATE + i  # Different seed per batch for variety
    
    for rdoi in chosen_reviews:
        rev_data = eval_data[eval_data['review_doi'] == rdoi]
        # Sample from each class within this review
        inc = rev_data[rev_data['label'] == 1].sample(n=N_INC_PER_REV, random_state=seed)
        exc = rev_data[rev_data['label'] == 0].sample(n=N_EXC_PER_REV, random_state=seed)
        batch_parts.append(pd.concat([inc, exc]))
    
    # Combine and shuffle all sampled papers
    batch = pd.concat(batch_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    batches.append(batch)
    
    print(
        f"Batch {i+1}: {len(batch)} papers | "
        f"{(batch['label']==1).sum()} INC / {(batch['label']==0).sum()} EXC | "
        f"{batch['review_doi'].nunique()} reviews"
    )

Reviews with enough papers (>=5 INC, >=20 EXC): 30
Chosen reviews: 8
Batch 1: 200 papers | 40 INC / 160 EXC | 8 reviews
Batch 2: 200 papers | 40 INC / 160 EXC | 8 reviews
Batch 3: 200 papers | 40 INC / 160 EXC | 8 reviews


---

## 3. Helper Functions

Core utility functions used throughout the evaluation pipeline:
- **`llm_call()`**: Unified inference supporting Ollama, OpenAI, and Google backends with rate limiting
- **`extract_decision()`**: Parse INCLUDE/EXCLUDE decisions from model responses
- **`compute_metrics()`**: Calculate F2 (primary), F1, recall, precision, accuracy
- **`create_prompt()`**: Format prompts with row data and truncation
- **`build_few_shot_prompt()`**: Inject in-context examples from the same review
- **`run_full_eval()`**: Full evaluation loop with caching and metrics computation

In [ ]:
# Unified LLM inference function supporting Ollama, OpenAI, and Google backends.
# Handles rate limiting with exponential backoff and returns response text with timing.

def llm_call(model_id, prompt, backend, system_prompt=None,
             temperature=TEMPERATURE, max_tokens=512):
    """
    Make an inference call to any supported LLM backend.
    
    Args:
        model_id: Model identifier (e.g., 'llama3.1:8b', 'gpt-4o-mini')
        prompt: User prompt text
        backend: One of 'ollama', 'openai', 'google'
        system_prompt: Optional system/instruction prompt
        temperature: Sampling temperature (0.0 = deterministic)
        max_tokens: Maximum response tokens
        
    Returns:
        Tuple of (response_text, elapsed_seconds)
    """
    t0, text = time.time(), ''

    # ─────────────────────────────────────────────────────────────────────────
    # Ollama backend (local models)
    # ─────────────────────────────────────────────────────────────────────────
    if backend == 'ollama':
        kw = dict(model=model_id, prompt=prompt,
                  options={'temperature': temperature,
                           'num_predict': max_tokens, 'num_gpu': 999})
        if system_prompt:
            kw['system'] = system_prompt
        text = ollama.generate(**kw).get('response', '')

    # ─────────────────────────────────────────────────────────────────────────
    # OpenAI backend (GPT models)
    # ─────────────────────────────────────────────────────────────────────────
    elif backend == 'openai':
        msgs = ([{'role': 'system', 'content': system_prompt}]
                if system_prompt else [])
        msgs.append({'role': 'user', 'content': prompt})
        
        # Retry logic with exponential backoff for rate limits
        for attempt in range(3):
            try:
                r = openai_client.chat.completions.create(
                    model=model_id, messages=msgs,
                    max_completion_tokens=max_tokens,
                    temperature=temperature)
                text = r.choices[0].message.content or ''
                break
            except openai.RateLimitError:
                time.sleep(30 * (attempt + 1))  # 30s, 60s, 90s backoff
            except Exception as e:
                text = f'ERROR: {e}'; break

    # ─────────────────────────────────────────────────────────────────────────
    # Google backend (Gemini models)
    # ─────────────────────────────────────────────────────────────────────────
    elif backend == 'google':
        config = genai_types.GenerateContentConfig(
            temperature=temperature, max_output_tokens=max_tokens)
        if system_prompt:
            config.system_instruction = system_prompt
        
        # Retry logic for rate limits
        for attempt in range(3):
            try:
                r = google_client.models.generate_content(
                    model=model_id, contents=prompt, config=config)
                text = r.text or ''
                break
            except Exception as e:
                if 'rate' in str(e).lower() or '429' in str(e):
                    time.sleep(30 * (attempt + 1))  # Exponential backoff
                else:
                    text = f'ERROR: {e}'; break
    else:
        raise ValueError(f'Unknown backend: {backend}')

    return text, time.time() - t0

In [ ]:
# Parse INCLUDE/EXCLUDE decision from model response text.
# Returns 1 (include), 0 (exclude), or -1 (invalid/unparseable response).
# Handles multiple response formats to maximize extraction success.

def extract_decision(response: str) -> int:
    """
    Extract binary classification decision from LLM response.
    
    Handles formats:
    - "DECISION: INCLUDE" or "DECISION: EXCLUDE"
    - Standalone "INCLUDE" or "EXCLUDE" on a line
    - "INCLUDE." or "EXCLUDE," at start of response
    - Natural language like "should be included"
    
    Args:
        response: Raw model response text
        
    Returns:
        1 (include), 0 (exclude), or -1 (unparseable)
    """
    if not response or not response.strip():
        return -1

    up = response.upper().strip()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Pattern 1: Explicit "DECISION: INCLUDE/EXCLUDE" format
    # ─────────────────────────────────────────────────────────────────────────
    m = re.search(r'(?m)^\s*DECISION\s*:\s*(INCLUDE|EXCLUDE)\b', up)
    if m:
        return 1 if m.group(1) == 'INCLUDE' else 0

    # ─────────────────────────────────────────────────────────────────────────
    # Pattern 2: Standalone keyword on a line
    # ─────────────────────────────────────────────────────────────────────────
    m = re.search(r'(?m)^\s*(INCLUDE|EXCLUDE)\s*$', up)
    if m:
        return 1 if m.group(1) == 'INCLUDE' else 0
    
    # ─────────────────────────────────────────────────────────────────────────
    # Pattern 3: Keyword at start of response (with punctuation)
    # ─────────────────────────────────────────────────────────────────────────
    m = re.match(r'\s*(INCLUDE|EXCLUDE)[.\s,]', up)
    if m:
        return 1 if m.group(1) == 'INCLUDE' else 0
    
    # ─────────────────────────────────────────────────────────────────────────
    # Pattern 4: Natural language phrasing ("should be included")
    # ─────────────────────────────────────────────────────────────────────────
    if re.search(r'\b(SHOULD|WOULD|WILL|MUST)\s+(BE\s+)?(INCLUDE|INCLUDED)\b', up):
        return 1
    if re.search(r'\b(SHOULD|WOULD|WILL|MUST)\s+(BE\s+)?(EXCLUDE|EXCLUDED)\b', up):
        return 0
    
    # Could not parse decision
    return -1

In [ ]:
# Compute classification metrics for valid predictions (excluding -1 invalid responses).
# Returns F2 (primary metric), F1, recall, precision, and accuracy.

def compute_metrics(labels, preds):
    """
    Calculate screening classification metrics.
    
    Args:
        labels: Ground truth labels (0/1)
        preds: Model predictions (0/1/-1 for invalid)
        
    Returns:
        Dictionary with valid count, invalid count, and metrics:
        - acc: Accuracy
        - prec: Precision
        - rec: Recall (sensitivity)
        - f1: F1 score
        - f2: F2 score (β=2, weights recall 4× over precision)
    """
    # Filter out invalid predictions (-1)
    mask = [p != -1 for p in preds]
    yt = [l for l, m in zip(labels, mask) if m]  # True labels for valid preds
    yp = [p for p, m in zip(preds, mask) if m]   # Valid predictions only
    
    n = sum(mask)    # Number of valid predictions
    inv = len(preds) - n  # Number of invalid predictions
    
    if n == 0:
        return dict(valid=0, invalid=inv, acc=0, prec=0, rec=0, f1=0, f2=0)
    
    return dict(
        valid   = n,
        invalid = inv,
        acc     = round(accuracy_score(yt, yp), 4),
        prec    = round(precision_score(yt, yp, zero_division=0), 4),
        rec     = round(recall_score(yt, yp, zero_division=0), 4),
        f1      = round(f1_score(yt, yp, zero_division=0), 4),
        f2      = round(fbeta_score(yt, yp, beta=2, zero_division=0), 4),  # Primary metric
    )

In [ ]:
# Print formatted metrics summary with confusion matrix breakdown.
# Reports valid prediction count, all metrics, and TP/FP/FN/TN counts.

def print_metrics(name, labels, preds):
    """
    Display formatted metrics summary with confusion matrix.
    
    Args:
        name: Display name for this evaluation run
        labels: Ground truth labels
        preds: Model predictions
        
    Returns:
        Metrics dictionary from compute_metrics()
    """
    m = compute_metrics(labels, preds)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Print summary line with key metrics
    # ─────────────────────────────────────────────────────────────────────────
    print(f"  {name}  ({m['valid']}/{len(labels)} valid)")
    print(f"    F2={m['f2']:.3f}  F1={m['f1']:.3f}  Rec={m['rec']:.3f}  "
          f"Prec={m['prec']:.3f}  Acc={m['acc']:.3f}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Print confusion matrix breakdown (TP, FP, FN, TN)
    # ─────────────────────────────────────────────────────────────────────────
    vt = [l for l, p in zip(labels, preds) if p != -1]
    vp = [p for p in preds if p != -1]
    if vt:
        cm = confusion_matrix(vt, vp)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            print(f"    TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    
    return m

In [ ]:
# Create formatted prompt from template by inserting row data with length truncation.
# Prevents context overflow by limiting each field to a reasonable character count.

def create_prompt(row, template, **kwargs):
    """
    Format a prompt template with paper/review data.
    
    Applies truncation to prevent excessive context length:
    - review_title: 500 chars
    - review_abstract: 2000 chars
    - objectives_text: 2500 chars
    - selection_criteria_text: 2500 chars
    - paper_title: 300 chars
    - paper_abstract: 2000 chars
    
    Args:
        row: DataFrame row with paper/review data
        template: Prompt template with {placeholders}
        **kwargs: Additional template variables
        
    Returns:
        Formatted prompt string
    """
    return template.format(
        review_title            = str(row.get('review_title', ''))[:500],
        review_abstract         = str(row.get('review_abstract', ''))[:2000],
        objectives_text         = str(row.get('objectives_text', ''))[:2500],
        selection_criteria_text = str(row.get('selection_criteria_text', ''))[:2500],
        paper_title             = str(row.get('paper_title', ''))[:300],
        paper_abstract          = str(row.get('paper_abstract', ''))[:2000],
        **kwargs,
    )

In [ ]:
# Build few-shot prompt with examples from the same review as the target paper.
# Samples n_inc included and n_exc excluded examples, ensuring the target paper is not included.

def build_few_shot_prompt(row, pool, template, n_inc=1, n_exc=1):
    """
    Create a few-shot prompt with in-context examples from the same review.
    
    Args:
        row: Target paper row to classify
        pool: Full dataset to draw examples from
        template: Prompt template with {examples} placeholder
        n_inc: Number of included paper examples to add
        n_exc: Number of excluded paper examples to add
        
    Returns:
        Formatted prompt with examples section populated
    """
    rdoi = row['review_doi']
    
    # ─────────────────────────────────────────────────────────────────────────
    # Find papers from the same review, excluding the target paper
    # ─────────────────────────────────────────────────────────────────────────
    same_review = pool[(pool['review_doi'] == rdoi) &
                       (pool['paper_title'] != row['paper_title'])]
    
    # ─────────────────────────────────────────────────────────────────────────
    # Sample examples from each class
    # ─────────────────────────────────────────────────────────────────────────
    inc = same_review[same_review['label'] == 1]
    exc = same_review[same_review['label'] == 0]
    inc_s = inc.sample(min(n_inc, len(inc)), random_state=RANDOM_STATE) if len(inc) > 0 else inc
    exc_s = exc.sample(min(n_exc, len(exc)), random_state=RANDOM_STATE) if len(exc) > 0 else exc

    # ─────────────────────────────────────────────────────────────────────────
    # Format examples as numbered text blocks
    # ─────────────────────────────────────────────────────────────────────────
    parts = []
    for i, (_, ex) in enumerate(pd.concat([inc_s, exc_s]).iterrows(), 1):
        label_str = 'INCLUDE' if ex['label'] == 1 else 'EXCLUDE'
        parts.append(
            f"Example {i} — {label_str}\n"
            f"Title: {str(ex['paper_title'])[:200]}\n"
            f"Abstract: {str(ex['paper_abstract'])[:500]}"
        )
    examples_text = '\n\n'.join(parts) if parts else '(no examples available)'
    
    return create_prompt(row, template, examples=examples_text)

In [ ]:
# Generate standardized file path for saving evaluation results by step, model, and variant.
# Format: results/{step}_{save_name}_{variant}.csv

def get_run_path(step, save_name, variant):
    """
    Generate path for saving/loading evaluation run results.
    
    Args:
        step: Pipeline phase (e.g., 'step1', 'step2', 'step3')
        save_name: Model save name (e.g., 'llama3_1_8b')
        variant: Configuration variant (e.g., 'p2_v1_baseline_obj_sel', '0shot')
        
    Returns:
        Path object to the results CSV file
    """
    return RESULTS_DIR / f'{step}_{save_name}_{variant}.csv'

In [ ]:
# Run full evaluation for a model on a dataset, with caching support.
# Loads cached results if available; otherwise runs inference on all rows and saves results.
# Returns metrics dictionary and prediction list.

def run_full_eval(data, model_id, save_name, backend,
                  prompt_fn, labels, step, variant,
                  system_prompt=None, max_tokens=512, desc=None):
    """
    Execute evaluation loop with caching and metrics computation.
    
    Args:
        data: DataFrame of papers to evaluate
        model_id: Model identifier for inference
        save_name: Model save name for results files
        backend: LLM backend ('ollama', 'openai', 'google')
        prompt_fn: Function to generate prompt from row
        labels: Ground truth labels
        step: Pipeline phase for file naming
        variant: Configuration variant for file naming
        system_prompt: Optional system prompt for supported backends
        max_tokens: Maximum response tokens
        desc: Progress bar description (auto-generated if None)
        
    Returns:
        Tuple of (metrics_dict, predictions_list)
    """
    run_path = get_run_path(step, save_name, variant)

    # ─────────────────────────────────────────────────────────────────────────
    # Check for cached results and load if available
    # ─────────────────────────────────────────────────────────────────────────
    if run_path.exists():
        prev  = pd.read_csv(run_path)
        preds = prev['prediction'].fillna(-1).astype(int).tolist()
        m = compute_metrics(labels, preds)
        m['time_min'] = round(prev['time_sec'].sum() / 60, 2)
        m['source'] = 'cached'
        print(f"  ✓ Loaded cached: {run_path.name}")
        print_metrics(f"{save_name} | {variant}", labels, preds)
        return m, preds

    # ─────────────────────────────────────────────────────────────────────────
    # Run fresh evaluation on all rows
    # ─────────────────────────────────────────────────────────────────────────
    preds, resps, times = [], [], []
    if desc is None:
        desc = f"{save_name} | {variant}"

    for _, row in tqdm(data.iterrows(), total=len(data), desc=desc):
        prompt = prompt_fn(row)
        try:
            resp, elapsed = llm_call(model_id, prompt, backend,
                                     system_prompt=system_prompt,
                                     max_tokens=max_tokens)
            pred = extract_decision(resp)
        except Exception as e:
            resp, pred, elapsed = str(e), -1, 0
        preds.append(pred)
        resps.append(resp)
        times.append(elapsed)

    # ─────────────────────────────────────────────────────────────────────────
    # Save results to CSV for caching
    # ─────────────────────────────────────────────────────────────────────────
    run_df = data[['review_doi', 'paper_title', 'label']].copy()
    run_df['prediction'] = preds
    run_df['response']   = resps
    run_df['time_sec']   = [round(t, 2) for t in times]
    run_df.to_csv(run_path, index=False)

    # ─────────────────────────────────────────────────────────────────────────
    # Compute and return metrics
    # ─────────────────────────────────────────────────────────────────────────
    m = compute_metrics(labels, preds)
    m['time_min'] = round(sum(times) / 60, 2)
    m['source'] = 'fresh'
    print_metrics(f"{save_name} | {variant}", labels, preds)
    return m, preds

---

## 4. Phase 1: Prompt Optimisation

Test 5 prompt variants across 3 stratified batches using all Ollama models to identify the best-performing prompt structure. Each variant uses Objectives + Selection Criteria context but differs in framing, system prompt usage, and instruction detail. **The winner is selected by highest mean F2 score.**

| Variant | Description |
|---------|-------------|
| V1 | Baseline with role framing |
| V2 | No role framing |
| V3 | System prompt separation |
| V4 | Minimal wording |
| V5 | Detailed step-by-step |

In [ ]:
# Define Phase 1 prompt variants for optimization.
# Tests 5 different prompt structures using Objectives + Selection Criteria context.
# Each variant takes a different approach to framing the screening task.

STEP1_PROMPT_VARIANTS = [

    # ─────────────────────────────────────────────────────────────────────────
    # V1: Baseline with role framing
    # Includes explicit "You are assisting with..." preamble
    # ─────────────────────────────────────────────────────────────────────────
    {
        "name": "p2_v1_baseline_obj_sel",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Using only the objectives and selection criteria above, should this paper be included in the review? In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # ─────────────────────────────────────────────────────────────────────────
    # V2: No role framing
    # Direct task without "You are..." preamble
    # ─────────────────────────────────────────────────────────────────────────
    {
        "name": "p2_v2_no_role_obj_sel",
        "system": None,
        "user": (
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Using only the objectives and selection criteria above, should this paper be included in the review? In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.\n\n"
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # ─────────────────────────────────────────────────────────────────────────
    # V3: With system prompt separation
    # Splits instructions into system prompt (role) + user prompt (task)
    # ─────────────────────────────────────────────────────────────────────────
    {
        "name": "p2_v3_with_system_prompt_obj_sel",
        "system": (
            "You are an expert medical librarian assisting with title/abstract screening "
            "for Cochrane systematic reviews. Use only the review objectives and "
            "selection criteria in the user prompt. Always respond with exactly one word: "
            "INCLUDE or EXCLUDE."
        ),
        "user": (
            "=== REVIEW CONTEXT ===\n"
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "=== CANDIDATE PAPER ===\n"
            "Title: {paper_title}\n\n"
            "Abstract: {paper_abstract}\n\n"
            "Should this paper be included in the review?\n\n"
            "In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable. Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # ─────────────────────────────────────────────────────────────────────────
    # V4: Minimal wording
    # Stripped-down prompt with essentials only
    # ─────────────────────────────────────────────────────────────────────────
    {
        "name": "p2_v4_minimal_obj_sel",
        "system": None,
        "user": (
            "Objectives: {objectives_text}\n\n"
            "Selection criteria: {selection_criteria_text}\n\n"
            "Paper title: {paper_title}\n"
            "Paper abstract: {paper_abstract}\n\n"
            "Use only the objectives and selection criteria. "
            "Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },

    # ─────────────────────────────────────────────────────────────────────────
    # V5: Detailed step-by-step
    # Explicit numbered steps for reasoning process
    # ─────────────────────────────────────────────────────────────────────────
    {
        "name": "p2_v5_detailed_obj_sel",
        "system": None,
        "user": (
            "You are assisting with title/abstract screening for a Cochrane systematic review.\n\n"
            "Step 1 — Understand the review scope from these fields only:\n"
            "  Objectives: {objectives_text}\n"
            "  Selection criteria: {selection_criteria_text}\n\n"
            "Step 2 — Evaluate the candidate paper:\n"
            "  Title: {paper_title}\n"
            "  Abstract: {paper_abstract}\n\n"
            "Step 3 — Decide whether the paper matches the review scope.\n"
            "Step 4 — Give one-word output.\n"
            "In this stage, it is critical not to miss any potentially relevant papers. False inclusions are acceptable and will be filtered later; false exclusions are not acceptable. Respond with exactly one word: INCLUDE or EXCLUDE"
        ),
    },
]

In [ ]:
# Execute Phase 1 prompt optimization across all batches, variants, and Ollama models.
# Evaluates 3 batches × 5 prompt variants × 4 Ollama models = 60 total runs.
# Saves individual run results to CSV files and aggregated metrics.

step1_rows = []
total = len(batches) * len(STEP1_PROMPT_VARIANTS) * len(OLLAMA_MODELS)
run_n = 0

# ─────────────────────────────────────────────────────────────────────────────
# Main evaluation loop: batch × variant × model
# ─────────────────────────────────────────────────────────────────────────────
for batch_idx, test_subset in enumerate(batches):
    labels = test_subset['label'].tolist()

    for variant in STEP1_PROMPT_VARIANTS:
        vname   = variant['name']
        tmpl    = variant['user']
        sys_msg = variant['system']

        for mid, save_name, backend in OLLAMA_MODELS:
            run_n += 1
            print(f"\n[{run_n}/{total}]  batch={batch_idx+1}  {vname}  |  {save_name}")

            # Run evaluation and collect metrics
            m, _ = run_full_eval(
                test_subset, mid, save_name, backend,
                lambda row, t=tmpl: create_prompt(row, t),
                labels,
                step='step1', variant=f'{vname}_B{batch_idx+1}',
                system_prompt=sys_msg,
            )
            
            # Store results for aggregation
            step1_rows.append({
                'batch': batch_idx + 1,
                'model': save_name,
                'prompt': vname,
                **m,
            })

# ─────────────────────────────────────────────────────────────────────────────
# Save aggregated Phase 1 results
# ─────────────────────────────────────────────────────────────────────────────
step1_df = pd.DataFrame(step1_rows)
step1_df.to_csv(RESULTS_DIR / 'step1_prompt_testing.csv', index=False)
print(f"\n{'='*60}\nStep 1 complete — {run_n} runs\n{'='*60}")


[1/60]  batch=1  p2_v1_baseline_obj_sel  |  llama3_1_8b


llama3_1_8b | p2_v1_baseline_obj_sel_B1:   0%|          | 0/200 [00:00<?, ?it/s]

  llama3_1_8b | p2_v1_baseline_obj_sel_B1  (200/200 valid)
    F2=0.759  F1=0.654  Rec=0.850  Prec=0.531  Acc=0.820
    TP=34  FP=30  FN=6  TN=130

[2/60]  batch=1  p2_v1_baseline_obj_sel  |  mixtral_8x7b


mixtral_8x7b | p2_v1_baseline_obj_sel_B1:   0%|          | 0/200 [00:00<?, ?it/s]

  mixtral_8x7b | p2_v1_baseline_obj_sel_B1  (163/200 valid)
    F2=0.714  F1=0.709  Rec=0.718  Prec=0.700  Acc=0.859
    TP=28  FP=12  FN=11  TN=112

[3/60]  batch=1  p2_v1_baseline_obj_sel  |  gemma3_27b


...

  mixtral_8x7b | p2_v5_detailed_obj_sel_B3  (175/200 valid)
    F2=0.556  F1=0.583  Rec=0.538  Prec=0.636  Acc=0.886
    TP=14  FP=8  FN=12  TN=141

[59/60]  batch=3  p2_v5_detailed_obj_sel  |  gemma3_27b


gemma3_27b | p2_v5_detailed_obj_sel_B3:   0%|          | 0/200 [00:00<?, ?it/s]

  gemma3_27b | p2_v5_detailed_obj_sel_B3  (200/200 valid)
    F2=0.727  F1=0.640  Rec=0.800  Prec=0.533  Acc=0.820
    TP=32  FP=28  FN=8  TN=132

[60/60]  batch=3  p2_v5_detailed_obj_sel  |  mistral_small3_2_24b


mistral_small3_2_24b | p2_v5_detailed_obj_sel_B3:   0%|          | 0/200 [00:00<?, ?it/s]

  mistral_small3_2_24b | p2_v5_detailed_obj_sel_B3  (200/200 valid)
    F2=0.672  F1=0.667  Rec=0.675  Prec=0.658  Acc=0.865
    TP=27  FP=14  FN=13  TN=146

Step 1 complete — 60 runs


In [ ]:
# Analyze Phase 1 results and select the winning prompt based on mean F2 score.
# Loads cached results and ranks prompts by F2 (primary metric) then recall.

# ─────────────────────────────────────────────────────────────────────────────
# Load Phase 1 results from CSV
# ─────────────────────────────────────────────────────────────────────────────
step1_df = pd.read_csv(RESULTS_DIR / 'step1_prompt_testing.csv')

# ─────────────────────────────────────────────────────────────────────────────
# Aggregate by prompt variant: mean across all batches and models
# ─────────────────────────────────────────────────────────────────────────────
step1_by_prompt = (
    step1_df.groupby('prompt')[['f2','f1','rec','prec','acc']]
    .mean().round(4).reset_index()
    .sort_values(['f2','rec'], ascending=False)  # Rank by F2, then recall
)

print("STEP 1 — ranked by mean F2 across batches & models")
print(step1_by_prompt.to_string(index=False))

print("\nPer-batch, per-model detail:")
print(step1_df[['batch','model','prompt','f2','rec','prec']].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# Extract winning prompt configuration
# ─────────────────────────────────────────────────────────────────────────────
WINNING_PROMPT_NAME    = step1_by_prompt.iloc[0]['prompt']
WINNING_PROMPT_VARIANT = next(v for v in STEP1_PROMPT_VARIANTS if v['name'] == WINNING_PROMPT_NAME)
WINNING_TEMPLATE       = WINNING_PROMPT_VARIANT['user']
WINNING_SYSTEM         = WINNING_PROMPT_VARIANT['system']

print(f"\n✓ WINNING PROMPT : {WINNING_PROMPT_NAME}")
print(f"  F2={step1_by_prompt.iloc[0]['f2']:.4f}  "
      f"Rec={step1_by_prompt.iloc[0]['rec']:.4f}  "
      f"Prec={step1_by_prompt.iloc[0]['prec']:.4f}")

STEP 1 — ranked by mean F2 across batches & models
                          prompt     f2     f1    rec   prec    acc
           p2_v2_no_role_obj_sel 0.7327 0.6077 0.8800 0.4905 0.7422
          p2_v1_baseline_obj_sel 0.7046 0.6566 0.7492 0.6078 0.8336
          p2_v5_detailed_obj_sel 0.6864 0.6548 0.7143 0.6219 0.8574
p2_v3_with_system_prompt_obj_sel 0.6735 0.6396 0.7110 0.6226 0.8389
           p2_v4_minimal_obj_sel 0.5643 0.5680 0.6102 0.6796 0.7965

Per-batch, per-model detail:
 batch                model                           prompt     f2    rec   prec
     1          llama3_1_8b           p2_v1_baseline_obj_sel 0.7589 0.8500 0.5312
     1         mixtral_8x7b           p2_v1_baseline_obj_sel 0.7143 0.7179 0.7000
     1           gemma3_27b           p2_v1_baseline_obj_sel 0.7727 0.8500 0.5667
     1 mistral_small3_2_24b           p2_v1_baseline_obj_sel 0.7254 0.7000 0.8485
     1          llama3_1_8b            p2_v2_no_role_obj_sel 0.6588 0.9750 0.2868
     1         mixt

---

## 5. Phase 2: Few-Shot Model Comparison

Using the winning prompt from Phase 1, test 0/1/2-shot configurations across all 7 models on a stratified 5,000-paper sample. Few-shot examples are drawn from the same review as the target paper to preserve context relevance. **The winning shot count and best models are identified by F2 score.**

In [ ]:
# Note: If running Step 2 separately, manually insert the winning prompt variant from Step 1.

In [ ]:
# Build the few-shot template by injecting an examples block into the winning prompt.
# Splits at the candidate paper section to position examples between context and target paper.

_SPLIT_TOKEN = "=== CANDIDATE PAPER ==="

# ─────────────────────────────────────────────────────────────────────────────
# Insert examples section before the candidate paper
# ─────────────────────────────────────────────────────────────────────────────
if _SPLIT_TOKEN in WINNING_TEMPLATE:
    # Standard case: split at candidate paper section
    _before, _after = WINNING_TEMPLATE.split(_SPLIT_TOKEN, 1)
    FEW_SHOT_TEMPLATE = (
        _before
        + "=== EXAMPLES OF CORRECTLY SCREENED PAPERS ===\n\n"
        + "{examples}\n\n"
        + "Based on the review context and examples above, should this paper be included?\n\n"
        + _SPLIT_TOKEN
        + _after
    )
else:
    # Fallback: split before last line (response instruction)
    *body, last_line = WINNING_TEMPLATE.strip().splitlines()
    FEW_SHOT_TEMPLATE = (
        "\n".join(body)
        + "\n\n=== EXAMPLES OF CORRECTLY SCREENED PAPERS ===\n\n"
        + "{examples}\n\n"
        + "Based on the review context and examples above, should this paper be included?\n\n"
        + last_line
    )

In [ ]:
# Verify that the winning prompt and few-shot template are correctly configured.
# Confirms template injection was successful for Phase 2 evaluation.

print("="*60)
print("WINNING PROMPT VERIFICATION")
print("="*60)

# Display winning configuration
print(f"WINNING_PROMPT_NAME:   {WINNING_PROMPT_NAME}")
print(f"WINNING_SYSTEM:        {WINNING_SYSTEM}")
print(f"\nWINNING_TEMPLATE (first 300 chars):")
print(WINNING_TEMPLATE[:300])

# Verify few-shot template was built correctly
print("\n" + "="*60)
print(f"FEW_SHOT_TEMPLATE built correctly: {'=== EXAMPLES OF CORRECTLY SCREENED PAPERS ===' in FEW_SHOT_TEMPLATE}")
print(f"Split token found in template: {'=== CANDIDATE PAPER ===' in WINNING_TEMPLATE}")

WINNING PROMPT VERIFICATION
WINNING_PROMPT_NAME:   p2_v2_no_role_obj_sel
WINNING_SYSTEM:        None

WINNING_TEMPLATE (first 300 chars):
=== REVIEW CONTEXT ===
Objectives: {objectives_text}

Selection criteria: {selection_criteria_text}

=== CANDIDATE PAPER ===
Title: {paper_title}

Abstract: {paper_abstract}

Using only the objectives and selection criteria above, should this paper be included in the review? In this stage, it is cri

FEW_SHOT_TEMPLATE built correctly: True
Split token found in template: True


In [ ]:
# Build stratified sample for Phase 2+ evaluation to reduce runtime while maintaining validity.
# Preserves review-level proportions and within-review label distributions.
# First deduplicates source data, then samples proportionally from each (review, label) stratum.

STEP2_SAMPLE_SIZE = 5000

def build_stratified_sample(data: pd.DataFrame, target_size: int, random_state: int = 42) -> pd.DataFrame:
    """
    Build a stratified sample that preserves:
    1. Review-level proportions (% of papers from each review)
    2. Within-review label proportions (% INCLUDE/EXCLUDE per review)
    
    Args:
        data: Full evaluation dataset
        target_size: Desired sample size
        random_state: Random seed for reproducibility
        
    Returns:
        Stratified sample DataFrame
    """
    data = data.copy()
    
    # ─────────────────────────────────────────────────────────────────────────
    # Create unique paper identifier for deduplication
    # ─────────────────────────────────────────────────────────────────────────
    data['paper_id'] = data['review_doi'].astype(str) + '|' + data['paper_title'].astype(str)
    
    original_count = len(data)
    data = data.drop_duplicates(subset='paper_id', keep='first')
    dedup_count = len(data)
    
    if original_count != dedup_count:
        print(f"  Deduplication: {original_count} → {dedup_count} rows ({original_count - dedup_count} duplicates removed)")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Sample proportionally from each (review, label) stratum
    # ─────────────────────────────────────────────────────────────────────────
    result_parts = []
    review_totals = data.groupby('review_doi').size()
    total_papers = len(data)
    
    for review_doi in review_totals.index:
        # Calculate review's target sample size (proportional to original)
        review_original_pct = review_totals[review_doi] / total_papers
        review_target_n = max(1, round(review_original_pct * target_size))
        
        review_data = data[data['review_doi'] == review_doi]
        
        # Calculate label distribution within this review
        inc_mask = review_data['label'] == 1
        n_inc_orig = inc_mask.sum()
        n_exc_orig = len(review_data) - n_inc_orig
        
        # ─────────────────────────────────────────────────────────────────────
        # Handle edge case: review has only one class
        # ─────────────────────────────────────────────────────────────────────
        if n_inc_orig == 0 or n_exc_orig == 0:
            sample_n = min(review_target_n, len(review_data))
            sampled = review_data.sample(n=sample_n, random_state=random_state)
        else:
            # Sample from each class proportionally
            inc_pct = n_inc_orig / len(review_data)
            n_inc_target = max(1, round(inc_pct * review_target_n))
            n_exc_target = max(1, review_target_n - n_inc_target)
            
            inc_data = review_data[inc_mask]
            exc_data = review_data[~inc_mask]
            
            inc_sample = inc_data.sample(n=min(n_inc_target, len(inc_data)), random_state=random_state)
            exc_sample = exc_data.sample(n=min(n_exc_target, len(exc_data)), random_state=random_state)
            
            sampled = pd.concat([inc_sample, exc_sample])
        
        result_parts.append(sampled)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Combine and shuffle final sample
    # ─────────────────────────────────────────────────────────────────────────
    result = pd.concat(result_parts, ignore_index=True)
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Build the Phase 2 sample
# ─────────────────────────────────────────────────────────────────────────────
step2_sample = build_stratified_sample(eval_data, STEP2_SAMPLE_SIZE, RANDOM_STATE)
step2_data = step2_sample.copy()

# Ensure paper_id exists for tracking
if 'paper_id' not in step2_data.columns:
    step2_data['paper_id'] = step2_data['review_doi'].astype(str) + '|' + step2_data['paper_title'].astype(str)

print(f"\nOriginal dataset:   {len(eval_data):,} rows ({eval_data['review_doi'].nunique()} reviews)")
print(f"Stratified sample:  {len(step2_data):,} papers ({step2_data['review_doi'].nunique()} reviews)")
print(f"Sample labels:      {step2_data['label'].value_counts().to_dict()}")

  Deduplication: 10225 → 10220 rows (5 duplicates removed)

Original dataset:   10,225 rows (30 reviews)
Stratified sample:  5,003 papers (30 reviews)
Sample labels:      {0: 3894, 1: 1109}


In [ ]:
# Validate review-level proportions are preserved in the stratified sample.
# Compares the percentage of papers from each review in original vs. sample datasets.

# ─────────────────────────────────────────────────────────────────────────────
# Calculate review proportions in both datasets
# ─────────────────────────────────────────────────────────────────────────────
orig_review_pct = (eval_data.groupby('review_doi').size() / len(eval_data) * 100).round(2)
sample_review_pct = (step2_data.groupby('review_doi').size() / len(step2_data) * 100).round(2)

# Build comparison table
review_comparison = pd.DataFrame({
    'original_%': orig_review_pct,
    'sample_%': sample_review_pct,
    'diff_%': (sample_review_pct - orig_review_pct).round(2)
}).sort_values('original_%', ascending=False)

# ─────────────────────────────────────────────────────────────────────────────
# Display validation results
# ─────────────────────────────────────────────────────────────────────────────
print("VALIDATION 1: Review-Level Proportions")
print("="*60)
print(f"Max percentage difference: {review_comparison['diff_%'].abs().max():.2f}%")
print(f"\nAll reviews preserved in sample: {set(eval_data['review_doi'].unique()) == set(step2_data['review_doi'].unique())}")
print("\nReview proportions comparison (top 10 by size):")
print(review_comparison.head(10).to_string())

VALIDATION 1: Review-Level Proportions
Max percentage difference: 0.01%

All reviews preserved in sample: True

Review proportions comparison (top 10 by size):
            original_%  sample_%  diff_%
review_doi                              
CD009905         12.70     12.69   -0.01
CD010919          8.22      8.22    0.00
CD010321          6.82      6.82    0.00
CD010246          6.58      6.58    0.00
CD007825          6.30      6.30    0.00
CD011779          4.53      4.54    0.01
CD012219          4.33      4.34    0.01
CD011218          4.32      4.32    0.00
CD011856          4.11      4.10   -0.01
CD008657          3.84      3.84    0.00


In [ ]:
# Validate within-review label proportions are preserved in the stratified sample.
# Compares the percentage of INCLUDE papers within each review in original vs. sample.

def get_within_review_label_pcts(df, label_col='label', review_col='review_doi'):
    """Calculate % INCLUDE for each review."""
    return df.groupby(review_col)[label_col].mean() * 100

# ─────────────────────────────────────────────────────────────────────────────
# Calculate INCLUDE percentages per review in both datasets
# ─────────────────────────────────────────────────────────────────────────────
orig_label_pct = get_within_review_label_pcts(eval_data)
sample_label_pct = get_within_review_label_pcts(step2_data)

# Build comparison table
label_comparison = pd.DataFrame({
    'original_INCLUDE_%': orig_label_pct.round(2),
    'sample_INCLUDE_%': sample_label_pct.round(2),
    'diff_%': (sample_label_pct - orig_label_pct).round(2)
}).sort_values('original_INCLUDE_%', ascending=False)

# ─────────────────────────────────────────────────────────────────────────────
# Display validation results
# ─────────────────────────────────────────────────────────────────────────────
print("VALIDATION 2: Within-Review Label Proportions (% INCLUDE)")
print("="*60)
print(f"Max label proportion difference: {label_comparison['diff_%'].abs().max():.2f}%")
print(f"Mean absolute difference: {label_comparison['diff_%'].abs().mean():.2f}%")
print("\nWithin-review label comparison (all reviews):")
print(label_comparison.to_string())

VALIDATION 2: Within-Review Label Proportions (% INCLUDE)
Max label proportion difference: 1.82%
Mean absolute difference: 0.23%

Within-review label comparison (all reviews):
            original_INCLUDE_%  sample_INCLUDE_%  diff_%
review_doi                                              
CD004333                 24.32             24.41    0.09
CD010321                 23.67             23.75    0.08
CD009905                 23.63             23.62   -0.01
CD010246                 23.33             23.40    0.08
CD010919                 23.33             23.36    0.02
CD002246                 22.78             22.83    0.05
CD008657                 22.65             22.92    0.27
CD009604                 22.28             22.22   -0.06
CD011779                 22.25             22.03   -0.22
CD001855                 22.12             21.82   -0.31
CD011856                 21.90             21.95    0.05
CD004334                 21.74             22.06    0.32
CD012415                 2

In [ ]:
# Define Phase 2 configuration for 0/1/2-shot evaluation.
# Tests different numbers of in-context examples with all models using the winning prompt.

# ─────────────────────────────────────────────────────────────────────────────
# Shot configurations: (name, n_included_examples, n_excluded_examples)
# ─────────────────────────────────────────────────────────────────────────────
SHOT_CONFIGS = [
    ('0shot', 0, 0),   # No examples (zero-shot)
    ('1shot', 1, 1),   # 1 included + 1 excluded example
    ('2shot', 2, 2),   # 2 included + 2 excluded examples
]

# Ensure paper_id for tracking across runs
step2_data['paper_id'] = step2_data['review_doi'].astype(str) + '|' + step2_data['paper_title'].astype(str)

# ─────────────────────────────────────────────────────────────────────────────
# Print configuration summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"Step 2 configuration:")
print(f"  - Sample size: {len(step2_data):,} papers")
print(f"  - Shot configs: {[s[0] for s in SHOT_CONFIGS]}")
print(f"  - Models: {len(ALL_MODELS)} ({[m[1] for m in ALL_MODELS]})")
print(f"  - Using winning prompt: {WINNING_PROMPT_NAME}")
print(f"  - Total runs: {len(SHOT_CONFIGS) * len(ALL_MODELS)}")

Step 2 configuration:
  - Sample size: 5,003 papers
  - Shot configs: ['0shot', '1shot', '2shot']
  - Models: 7 (['llama3_1_8b', 'mixtral_8x7b', 'gemma3_27b', 'mistral_small3_2_24b', 'biomistral', 'gpt_4o_mini', 'gemini_2_5_flash'])
  - Using winning prompt: p2_v2_no_role_obj_sel
  - Total runs: 21


In [ ]:
# Execute Phase 2: 0/1/2-shot evaluation on the stratified sample.
# Tests all 7 models × 3 shot configurations = 21 runs using the winning prompt from Phase 1.
# Loads cached results when available, otherwise runs fresh inference.

step2_rows  = []
step2_preds = {}

labels = step2_data['label'].tolist()
total = len(ALL_MODELS) * len(SHOT_CONFIGS)
run_n = 0

print(f"Starting Step 2 evaluation")
print(f"Sample size: {len(step2_data):,} papers")
print(f"Winning prompt: {WINNING_PROMPT_NAME}")
print("="*60)

# ─────────────────────────────────────────────────────────────────────────────
# Main evaluation loop: shot configuration × model
# ─────────────────────────────────────────────────────────────────────────────
for shot_name, n_inc, n_exc in SHOT_CONFIGS:
    for mid, save_name, backend in ALL_MODELS:
        run_n += 1
        run_path = get_run_path('step2', save_name, shot_name)
        
        print(f"\n[{run_n}/{total}]  {shot_name}  |  {save_name}")
        
        # ─────────────────────────────────────────────────────────────────────
        # Check for cached results
        # ─────────────────────────────────────────────────────────────────────
        if run_path.exists():
            print(f"  ✓ Loading existing: {run_path.name}")
            existing_df = pd.read_csv(run_path)
            preds = existing_df['prediction'].tolist()
            m = compute_metrics(labels, preds)
            m['time_min'] = round(existing_df['time_sec'].sum() / 60, 2) if 'time_sec' in existing_df else 0
            m['source'] = 'loaded from CSV'
            print_metrics(f"{save_name} | {shot_name}", labels, preds)
            step2_preds[(save_name, shot_name)] = preds
            step2_rows.append({
                'model': save_name,
                'shot': shot_name,
                **m,
            })
        else:
            # ─────────────────────────────────────────────────────────────────
            # Run fresh evaluation
            # ─────────────────────────────────────────────────────────────────
            print(f"  → Running fresh evaluation...")
            
            # Choose prompt function based on shot configuration
            if shot_name == '0shot':
                pfn = lambda row, t=WINNING_TEMPLATE: create_prompt(row, t)
                sys = WINNING_SYSTEM
            else:
                # Few-shot: build prompt with examples from same review
                def pfn(row, ni=n_inc, ne=n_exc):
                    return build_few_shot_prompt(row, eval_data, FEW_SHOT_TEMPLATE, ni, ne)
                sys = None

            m, preds = run_full_eval(
                step2_data, mid, save_name, backend,
                pfn, labels,
                step='step2', variant=f'{shot_name}',
                system_prompt=sys,
            )
            step2_preds[(save_name, shot_name)] = preds
            step2_rows.append({
                'model': save_name,
                'shot': shot_name,
                **m,
            })

# ─────────────────────────────────────────────────────────────────────────────
# Save aggregated Phase 2 metrics
# ─────────────────────────────────────────────────────────────────────────────
step2_metrics_df = pd.DataFrame(step2_rows)
step2_metrics_df.to_csv(RESULTS_DIR / 'step2_metrics.csv', index=False)
print(f"\n{'='*60}\nStep 2 complete — {run_n} runs on {len(step2_data):,} papers\n{'='*60}")

Starting Step 2 evaluation
Sample size: 5,003 papers
Winning prompt: p2_v2_no_role_obj_sel

[1/21]  0shot  |  llama3_1_8b
  ✓ Loading existing: step2_llama3_1_8b_0shot.csv
  llama3_1_8b | 0shot  (5003/5003 valid)
    F2=0.639  F1=0.419  Rec=0.985  Prec=0.266  Acc=0.395
    TP=1092  FP=3011  FN=17  TN=883

[2/21]  0shot  |  mixtral_8x7b
  ✓ Loading existing: step2_mixtral_8x7b_0shot.csv
  mixtral_8x7b | 0shot  (4768/5003 valid)
    F2=0.712  F1=0.581  Rec=0.838  Prec=0.444  Acc=0.735
    TP=876  FP=1097  FN=169  TN=2626

[3/21]  0shot  |  gemma3_27b
  ✓ Loading existing: step2_gemma3_27b_0shot.csv
  gemma3_27b | 0shot  (5003/5003 valid)
    F2=0.719  F1=0.535  Rec=0.934  Prec=0.374  Acc=0.640
    TP=1036  FP=1730  FN=73  TN=2164

[4/21]  0shot  |  mistral_small3_2_24b
  ✓ Loading existing: step2_mistral_small3_2_24b_0shot.csv
  mistral_small3_2_24b | 0shot  (5003/5003 valid)
    F2=0.741  F1=0.654  Rec=0.812  Prec=0.548  Acc=0.810
    TP=901  FP=744  FN=208  TN=3150

[5/21]  0shot  |  m

medgemma_4b | 0shot:   0%|          | 0/5003 [00:00<?, ?it/s]

  medgemma_4b | 0shot  (5003/5003 valid)
    F2=0.584  F1=0.607  Rec=0.570  Prec=0.649  Acc=0.837
    TP=632  FP=341  FN=477  TN=3553

[6/21]  0shot  |  gpt_4o_mini
  ✓ Loading existing: step2_gpt_4o_mini_0shot.csv
  gpt_4o_mini | 0shot  (5002/5003 valid)
    F2=0.600  F1=0.635  Rec=0.579  Prec=0.703  Acc=0.853
    TP=641  FP=271  FN=467  TN=3623

[7/21]  0shot  |  gemini_2_5_flash
  ✓ Loading existing: step2_gemini_2_5_flash_0shot.csv
  gemini_2_5_flash | 0shot  (5002/5003 valid)
    F2=0.646  F1=0.642  Rec=0.648  Prec=0.636  Acc=0.840
    TP=719  FP=412  FN=390  TN=3481

[8/21]  1shot  |  llama3_1_8b
  ✓ Loading existing: step2_llama3_1_8b_1shot.csv
  llama3_1_8b | 1shot  (5003/5003 valid)
    F2=0.617  F1=0.396  Rec=0.982  Prec=0.248  Acc=0.337
    TP=1089  FP=3296  FN=20  TN=598

[9/21]  1shot  |  mixtral_8x7b
  ✓ Loading existing: step2_mixtral_8x7b_1shot.csv
  mixtral_8x7b | 1shot  (4302/5003 valid)
    F2=0.665  F1=0.471  Rec=0.917  Prec=0.317  Acc=0.587
    TP=791  FP=1706  FN=

medgemma_4b | 1shot:   0%|          | 0/5003 [00:00<?, ?it/s]

  medgemma_4b | 1shot  (5003/5003 valid)
    F2=0.700  F1=0.568  Rec=0.828  Prec=0.432  Acc=0.721
    TP=918  FP=1205  FN=191  TN=2689

[13/21]  1shot  |  gpt_4o_mini
  ✓ Loading existing: step2_gpt_4o_mini_1shot.csv
  gpt_4o_mini | 1shot  (5003/5003 valid)
    F2=0.685  F1=0.644  Rec=0.715  Prec=0.585  Acc=0.825
    TP=793  FP=562  FN=316  TN=3332

[14/21]  1shot  |  gemini_2_5_flash
  ✓ Loading existing: step2_gemini_2_5_flash_1shot.csv
  gemini_2_5_flash | 1shot  (4999/5003 valid)
    F2=0.600  F1=0.647  Rec=0.572  Prec=0.745  Acc=0.862
    TP=634  FP=217  FN=474  TN=3674

[15/21]  2shot  |  llama3_1_8b
  ✓ Loading existing: step2_llama3_1_8b_2shot.csv
  llama3_1_8b | 2shot  (5003/5003 valid)
    F2=0.627  F1=0.409  Rec=0.973  Prec=0.259  Acc=0.376
    TP=1079  FP=3092  FN=30  TN=802

[16/21]  2shot  |  mixtral_8x7b
  ✓ Loading existing: step2_mixtral_8x7b_2shot.csv
  mixtral_8x7b | 2shot  (4406/5003 valid)
    F2=0.664  F1=0.475  Rec=0.903  Prec=0.322  Acc=0.594
    TP=810  FP=1703

...

  medgemma_4b | 2shot  (5003/5003 valid)
    F2=0.697  F1=0.534  Rec=0.875  Prec=0.384  Acc=0.661
    TP=970  FP=1555  FN=139  TN=2339

[20/21]  2shot  |  gpt_4o_mini
  → Running fresh evaluation...


gpt_4o_mini | 2shot:   0%|          | 0/5003 [00:00<?, ?it/s]

  gpt_4o_mini | 2shot  (5003/5003 valid)
    F2=0.695  F1=0.643  Rec=0.735  Prec=0.572  Acc=0.820
    TP=815  FP=609  FN=294  TN=3285

[21/21]  2shot  |  gemini_2_5_flash
  → Running fresh evaluation...


gemini_2_5_flash | 2shot:   0%|          | 0/5003 [00:00<?, ?it/s]

  gemini_2_5_flash | 2shot  (4980/5003 valid)
    F2=0.606  F1=0.649  Rec=0.580  Prec=0.738  Acc=0.862
    TP=638  FP=226  FN=463  TN=3653

Step 2 complete — 21 runs on 5,003 papers


In [ ]:
# Analyze Phase 2 results and select the winning shot configuration based on mean F2 score.
# Compares 0-shot, 1-shot, and 2-shot across all models.

# ─────────────────────────────────────────────────────────────────────────────
# Load Phase 2 metrics
# ─────────────────────────────────────────────────────────────────────────────
step2_metrics_df = pd.read_csv(RESULTS_DIR / 'step2_metrics.csv')

# ─────────────────────────────────────────────────────────────────────────────
# Aggregate by shot configuration: mean across all models
# ─────────────────────────────────────────────────────────────────────────────
shot_summary = (
    step2_metrics_df.groupby('shot')[['f2','rec','prec']]
    .mean().round(4).reset_index()
    .sort_values(['f2','rec'], ascending=False)  # Rank by F2, then recall
)

print("STEP 2 — Mean F2 by shot count (all 5 models)")
print(shot_summary.to_string(index=False))

print("\nFull per-model, per-shot:")
print(step2_metrics_df[['model','shot','f2','rec','prec','acc']].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# Select winning shot configuration
# ─────────────────────────────────────────────────────────────────────────────
WINNING_SHOT = shot_summary.iloc[0]['shot']
print(f"\n✓ WINNING SHOT : {WINNING_SHOT}")

STEP 2 — Mean F2 by shot count (all 5 models)
 shot     f2    rec   prec
2shot 0.6810 0.8168 0.4753
1shot 0.6733 0.8037 0.4802
0shot 0.6630 0.7666 0.5172

Full per-model, per-shot:
               model  shot     f2    rec   prec    acc
         llama3_1_8b 0shot 0.6394 0.9847 0.2661 0.3948
        mixtral_8x7b 0shot 0.7118 0.8383 0.4440 0.7345
          gemma3_27b 0shot 0.7192 0.9342 0.3745 0.6396
mistral_small3_2_24b 0shot 0.7408 0.8124 0.5477 0.8097
         medgemma_4b 0shot 0.5842 0.5699 0.6495 0.8365
         gpt_4o_mini 0shot 0.5997 0.5785 0.7029 0.8525
    gemini_2_5_flash 0shot 0.6458 0.6483 0.6357 0.8397
         llama3_1_8b 1shot 0.6173 0.9820 0.2483 0.3372
        mixtral_8x7b 1shot 0.6648 0.9166 0.3168 0.5867
          gemma3_27b 1shot 0.7293 0.8377 0.4806 0.7633
mistral_small3_2_24b 1shot 0.7170 0.7746 0.5528 0.8111
         medgemma_4b 1shot 0.6998 0.8278 0.4324 0.7210
         gpt_4o_mini 1shot 0.6847 0.7151 0.5852 0.8245
    gemini_2_5_flash 1shot 0.6000 0.5722 0.7450 0

In [ ]:
# Build per-paper classification table for council approach and inter-rater analysis.
# Creates a table with columns: review_doi, paper_title, label, and one column per model.

# ─────────────────────────────────────────────────────────────────────────────
# Get list of all model save names
# ─────────────────────────────────────────────────────────────────────────────
all_save_names = [m[1] for m in ALL_MODELS]

# Initialize table with paper identifiers and ground truth
step2_table = step2_sample[['review_doi', 'paper_title', 'label']].copy()

# ─────────────────────────────────────────────────────────────────────────────
# Load predictions for each model (from CSV or memory)
# ─────────────────────────────────────────────────────────────────────────────
for save_name in all_save_names:
    run_path = get_run_path('step2', save_name, WINNING_SHOT)

    if run_path.exists():
        # Load from cached CSV
        step2_table[save_name] = pd.read_csv(run_path)['prediction'].values
    elif (save_name, WINNING_SHOT) in step2_preds:
        # Load from in-memory predictions
        step2_table[save_name] = step2_preds[(save_name, WINNING_SHOT)]
    else:
        print(f"  ⚠ No predictions for {save_name} / {WINNING_SHOT}")
        step2_table[save_name] = -1

# ─────────────────────────────────────────────────────────────────────────────
# Save classification table for council analysis
# ─────────────────────────────────────────────────────────────────────────────
step2_table.to_csv(RESULTS_DIR / 'step2_classifications.csv', index=False)
print(f"\n✓ Saved: step2_classifications.csv  {step2_table.shape}")
display(step2_table.head())


✓ Saved: step2_classifications.csv  (5003, 10)


,review_doi,paper_title,label,llama3_1_8b,mixtral_8x7b,gemma3_27b,mistral_small3_2_24b,medgemma_4b,gpt_4o_mini,gemini_2_5_flash
0,CD004334,Evaluation of a community-wide incentive progr...,1,1,1,1,1,1,1,0
1,CD009905,Six-year sustainability of evidence-based inte...,1,1,-1,1,1,0,0,1
2,CD007825,The Fort McMurray Demonstration Project in Soc...,1,1,0,1,0,1,1,1
3,CD008160,Ergonomics and work organization: the relation...,0,1,1,1,1,1,1,0
4,CD012219,Indicators of hearing protection use: self-rep...,0,1,0,0,0,0,0,0


---

## 6. Phase 3: CoT & Reasoning

Evaluate whether explicit reasoning improves classification:
- **Standard models** (Llama, Mixtral, Gemma, Mistral, MedGemma): Use **Framework CoT** with PICO-based reasoning (Population, Intervention, Comparison, Outcome)
- **Reasoning models** (GPT-4o-mini, Gemini 2.5 Flash): Use **native reasoning** leveraging built-in deliberation capabilities

Evaluated on a 2,000-paper stratified subsample from Phase 2.

In [ ]:
# Step 3 uses Framework CoT (PICO-based reasoning) for standard models and native reasoning for GPT/Gemini.
# Both prompts are built from the winning Step 2 configuration.

In [ ]:
# Construct Phase 3 base prompt from winning Phase 2 configuration (0-shot or few-shot template).
# Extracts the paper section for reinsertion after adding reasoning instructions.

# ─────────────────────────────────────────────────────────────────────────────
# Select base template based on winning shot configuration
# ─────────────────────────────────────────────────────────────────────────────
_STEP3_BASE = WINNING_TEMPLATE if WINNING_SHOT == '0shot' else FEW_SHOT_TEMPLATE
_SPLIT_TOKEN = "=== CANDIDATE PAPER ==="

# ─────────────────────────────────────────────────────────────────────────────
# Extract the paper section for reinsertion into reasoning prompts
# ─────────────────────────────────────────────────────────────────────────────
if _SPLIT_TOKEN in _STEP3_BASE:
    _before, _after = _STEP3_BASE.split(_SPLIT_TOKEN, 1)
    # Remove the "Respond with..." instruction to add reasoning steps instead
    _paper_section = _SPLIT_TOKEN + _after.rsplit("Respond with", 1)[0]
else:
    _before = _STEP3_BASE
    _paper_section = (
        "=== CANDIDATE PAPER ===\n"
        "Title: {paper_title}\n\n"
        "Abstract: {paper_abstract}\n\n"
    )

In [ ]:
# Define Framework CoT prompt using PICO (Population, Intervention, Comparison, Outcome) structure.
# Used for standard models (Llama, Mixtral, Gemma, Mistral, MedGemma).
# PICO is a standard evidence-based medicine framework for structuring clinical questions.

COT_PROMPT = (
    _before
    + _paper_section
    + "Evaluate this paper using the PICO framework:\n"
    + "  P (Population)   — Does the paper study the same population or condition specified in the objectives?\n"
    + "  I (Intervention) — Does the paper evaluate the intervention or exposure required by the selection criteria?\n"
    + "  C (Comparison)   — Does the paper include a relevant comparison group if required?\n"
    + "  O (Outcome)      — Does the paper report outcomes relevant to the review question?\n\n"
    + "Also consider: Does the study design meet the selection criteria?\n\n"
    + "In this stage, it is critical not to miss any potentially relevant papers. "
    + "False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.\n\n"
    + "After reasoning through each PICO element, respond with exactly one word: INCLUDE or EXCLUDE"
)

In [ ]:
# Define native reasoning prompt for proprietary models (GPT-4o-mini, Gemini 2.5 Flash).
# Leverages built-in reasoning capabilities without explicit framework structure.
# These models have internal chain-of-thought capabilities that benefit from open-ended prompting.

REASONING_PROMPT = (
    _before
    + _paper_section
    + "Carefully reason about whether this paper meets the review's objectives and selection criteria. "
    + "In this stage, it is critical not to miss any potentially relevant papers. "
    + "False inclusions are acceptable and will be filtered later; false exclusions are not acceptable.\n\n"
    + "Respond with exactly one word: INCLUDE or EXCLUDE"
)

In [ ]:
# Execute Phase 3: CoT/Reasoning evaluation on a 2,000-paper stratified subsample.
# Standard models use PICO-framework CoT; reasoning models (GPT, Gemini) use native reasoning.
# Validates stratification consistency between Phase 2 and Phase 3 samples.

# ─────────────────────────────────────────────────────────────────────────────
# Build stratified subsample for Phase 3 (smaller for computational efficiency)
# ─────────────────────────────────────────────────────────────────────────────
STEP3_SAMPLE_SIZE = 2000
sampling_frac = STEP3_SAMPLE_SIZE / len(step2_sample)

step3_parts = []
for (rdoi, label), grp in step2_sample.groupby(['review_doi', 'label']):
    # Sample proportionally from each (review, label) stratum
    n_take = max(1, int(round(len(grp) * sampling_frac)))
    n_take = min(n_take, len(grp))
    step3_parts.append(grp.sample(n=n_take, random_state=RANDOM_STATE))

step3_sample = pd.concat(step3_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Step 3 sample: {len(step3_sample):,} papers from {step3_sample['review_doi'].nunique()} reviews")
print(f"  Labels: {(step3_sample['label']==1).sum()} INC / {(step3_sample['label']==0).sum()} EXC")

# ─────────────────────────────────────────────────────────────────────────────
# Validate stratification consistency (Phase 2 → Phase 3)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("VALIDATION: Stratification check (Step 2 → Step 3)")
print(f"{'='*60}")

# Review distribution validation
step2_review_pct = step2_sample['review_doi'].value_counts(normalize=True).sort_index() * 100
step3_review_pct = step3_sample['review_doi'].value_counts(normalize=True).sort_index() * 100
review_comp = pd.DataFrame({
    'step2_pct': step2_review_pct,
    'step3_pct': step3_review_pct,
    'diff': (step3_review_pct - step2_review_pct).abs()
}).round(2)
print(f"\nReview distribution (% of papers per review):")
print(review_comp.to_string())
print(f"  Max absolute diff: {review_comp['diff'].max():.2f}%")

# Label distribution validation
step2_label_pct = step2_sample['label'].value_counts(normalize=True).sort_index() * 100
step3_label_pct = step3_sample['label'].value_counts(normalize=True).sort_index() * 100
label_comp = pd.DataFrame({
    'step2_pct': step2_label_pct,
    'step3_pct': step3_label_pct,
    'diff': (step3_label_pct - step2_label_pct).abs()
}).round(2)
label_comp.index = ['EXCLUDE', 'INCLUDE']
print(f"\nLabel distribution:")
print(label_comp.to_string())
print(f"  Max absolute diff: {label_comp['diff'].max():.2f}%")

# Per-review INCLUDE rate validation
print(f"\nPer-review label % (INC rate):")
step2_inc_rate = step2_sample.groupby('review_doi')['label'].mean() * 100
step3_inc_rate = step3_sample.groupby('review_doi')['label'].mean() * 100
inc_rate_comp = pd.DataFrame({
    'step2_inc%': step2_inc_rate,
    'step3_inc%': step3_inc_rate,
    'diff': (step3_inc_rate - step2_inc_rate).abs()
}).round(2)
print(inc_rate_comp.to_string())
print(f"  Max absolute diff: {inc_rate_comp['diff'].max():.2f}%")
print(f"{'='*60}\n")

# ─────────────────────────────────────────────────────────────────────────────
# Initialize evaluation tracking
# ─────────────────────────────────────────────────────────────────────────────
step3_rows  = []
step3_preds = {}
reasoning_save_names = [s for _, s, _ in REASONING_MODELS]
step3_labels = step3_sample['label'].tolist()

# Get example counts from winning shot configuration
shot_to_examples = {'0shot': (0, 0), '1shot': (1, 0), '2shot': (1, 1)}
n_inc, n_exc = shot_to_examples.get(WINNING_SHOT, (1, 1))


def build_examples_text(row, pool, n_inc, n_exc):
    """Build the examples text for few-shot prompts."""
    rdoi = row['review_doi']
    same_review = pool[(pool['review_doi'] == rdoi) &
                       (pool['paper_title'] != row['paper_title'])]
    inc = same_review[same_review['label'] == 1]
    exc = same_review[same_review['label'] == 0]
    inc_s = inc.sample(min(n_inc, len(inc)), random_state=RANDOM_STATE) if len(inc) > 0 else inc
    exc_s = exc.sample(min(n_exc, len(exc)), random_state=RANDOM_STATE) if len(exc) > 0 else exc
    
    parts = []
    for i, (_, ex) in enumerate(pd.concat([inc_s, exc_s]).iterrows(), 1):
        label_str = 'INCLUDE' if ex['label'] == 1 else 'EXCLUDE'
        parts.append(
            f"Example {i} — {label_str}\n"
            f"Title: {str(ex['paper_title'])[:200]}\n"
            f"Abstract: {str(ex['paper_abstract'])[:500]}"
        )
    return '\n\n'.join(parts) if parts else '(no examples available)'


def build_step3_prompt(row, template, n_inc, n_exc):
    """Build Phase 3 prompt with examples drawn from full eval_data for diversity."""
    examples_text = build_examples_text(row, eval_data, n_inc, n_exc)
    return template.format(
        review_title            = str(row.get('review_title', ''))[:500],
        review_abstract         = str(row.get('review_abstract', ''))[:2000],
        objectives_text         = str(row.get('objectives_text', ''))[:2500],
        selection_criteria_text = str(row.get('selection_criteria_text', ''))[:2500],
        paper_title             = str(row.get('paper_title', ''))[:300],
        paper_abstract          = str(row.get('paper_abstract', ''))[:2000],
        examples                = examples_text,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Execute Phase 3 evaluation for all models
# ─────────────────────────────────────────────────────────────────────────────
print(f"Starting Step 3 evaluation on {len(step3_sample):,} papers (stratified from Step 2 sample)")
print("="*60)

for mid, save_name, backend in ALL_MODELS:
    # Select prompt type based on model category
    is_reasoning = save_name in reasoning_save_names
    tmpl         = REASONING_PROMPT if is_reasoning else COT_PROMPT
    variant      = 'reasoning' if is_reasoning else 'cot'
    max_tok      = 2048 if is_reasoning else 1024  # More tokens for reasoning

    print(f"\n  {save_name}  [{variant}]")
    m, preds = run_full_eval(
        step3_sample, mid, save_name, backend,
        lambda row, t=tmpl, ni=n_inc, ne=n_exc: build_step3_prompt(row, t, ni, ne),
        step3_labels,
        step='step3', variant=variant,
        max_tokens=max_tok,
    )
    step3_preds[save_name] = preds
    step3_rows.append({'model': save_name, 'variant': variant, **m})

# ─────────────────────────────────────────────────────────────────────────────
# Save aggregated Phase 3 metrics
# ─────────────────────────────────────────────────────────────────────────────
step3_metrics_df = pd.DataFrame(step3_rows)
step3_metrics_df.to_csv(RESULTS_DIR / 'step3_metrics.csv', index=False)
print(f"\n{'='*60}\nStep 3 complete — {len(step3_rows)} runs on {len(step3_sample):,} papers\n{'='*60}")

Step 3 sample: 1,997 papers from 30 reviews
  Labels: 442 INC / 1555 EXC

VALIDATION: Stratification check (Step 2 → Step 3)

Review distribution (% of papers per review):
            step2_pct  step3_pct  diff
review_doi                            
CD001855         1.10       1.10  0.00
CD002246         2.54       2.55  0.02
CD003110         0.44       0.45  0.01
CD003985         1.50       1.50  0.00
CD004333         2.54       2.50  0.03
CD004334         1.36       1.35  0.01
CD004726         1.04       1.00  0.04
CD004728         0.30       0.30  0.00
CD006527         0.88       0.90  0.02
CD007825         6.30       6.31  0.01
CD008160         3.14       3.15  0.02
CD008657         3.84       3.86  0.02
CD008931         1.98       1.95  0.03
CD009467         2.44       2.40  0.03
CD009604         3.78       3.81  0.03
CD009820         2.70       2.70  0.01
CD009905        12.69      12.72  0.03
CD010067         1.28       1.25  0.03
CD010246         6.58       6.61  0.03
CD010321 

medgemma_4b | cot:   0%|          | 0/1997 [00:00<?, ?it/s]

  medgemma_4b | cot  (1997/1997 valid)
    F2=0.683  F1=0.515  Rec=0.873  Prec=0.365  Acc=0.635
    TP=386  FP=672  FN=56  TN=883

  gpt_4o_mini  [SKIPPED - assigned to other team member]

  gemini_2_5_flash  [SKIPPED - assigned to other team member]

Step 3 complete — 4 runs on 1,997 papers


In [ ]:
# Display Phase 3 CoT/Reasoning evaluation results.
# Compares PICO-framework CoT (standard models) vs. native reasoning (GPT, Gemini).

# ─────────────────────────────────────────────────────────────────────────────
# Load and display Phase 3 metrics
# ─────────────────────────────────────────────────────────────────────────────
step3_metrics_df = pd.read_csv(RESULTS_DIR / 'step3_metrics.csv')

print("STEP 3 — CoT / Reasoning results")
print(step3_metrics_df[['model','variant','f2','rec','prec','acc']].to_string(index=False))

In [ ]:
# Build per-paper classification table for Phase 3 results (CoT/Reasoning predictions).
# Same structure as Phase 2: review_doi, paper_title, label, and one column per model.

# Initialize table with paper identifiers and ground truth
step3_table = eval_data[['review_doi', 'paper_title', 'label']].copy()

# ─────────────────────────────────────────────────────────────────────────────
# Load predictions for each model
# ─────────────────────────────────────────────────────────────────────────────
for mid, save_name, backend in ALL_MODELS:
    # Determine variant name based on model type
    variant  = 'reasoning' if save_name in reasoning_save_names else 'cot'
    run_path = get_run_path('step3', save_name, variant)
    
    if run_path.exists():
        # Load from cached CSV
        step3_table[save_name] = pd.read_csv(run_path)['prediction'].values
    elif save_name in step3_preds:
        # Load from in-memory predictions
        step3_table[save_name] = step3_preds[save_name]
    else:
        print(f"  ⚠ No predictions for {save_name}")
        step3_table[save_name] = -1

# ─────────────────────────────────────────────────────────────────────────────
# Save classification table for council analysis
# ─────────────────────────────────────────────────────────────────────────────
step3_table.to_csv(RESULTS_DIR / 'step3_classifications.csv', index=False)
print(f"\n✓ Saved: step3_classifications.csv  {step3_table.shape}")
display(step3_table.head())

---

## 7. Phase 4: Council Approach (Multi-Agent)

Apply **majority voting** across all 7 models to combine predictions from Phases 2 and 3. Ties favor INCLUDE (recall-oriented). Computes inter-rater agreement metrics:
- **Cohen's kappa**: Pairwise agreement between reasoning models (GPT vs. Gemini)
- **Fleiss' kappa**: Multi-rater agreement across all 7 models

This approach tests whether model consensus improves screening accuracy compared to individual models.

In [ ]:
# Define helper functions for Phase 4 council approach and inter-rater agreement analysis.
# Includes majority voting, Cohen's kappa (pairwise), and Fleiss' kappa (multi-rater).

from sklearn.metrics import cohen_kappa_score


def majority_vote(row_preds):
    """
    Return classification via majority vote across models.
    
    Args:
        row_preds: List of predictions from each model (0, 1, or -1 for invalid)
        
    Returns:
        1 (INCLUDE) if majority or tie (recall-oriented)
        0 (EXCLUDE) if strict minority
        -1 if all predictions invalid
    """
    valid = [p for p in row_preds if p != -1]
    if not valid:
        return -1
    # Ties favor INCLUDE (recall-oriented: don't miss relevant papers)
    return 1 if sum(v == 1 for v in valid) >= len(valid) / 2 else 0


def council_metrics(table, model_cols, labels, label='council'):
    """
    Apply majority vote across models and compute metrics.
    
    Args:
        table: DataFrame with model predictions as columns
        model_cols: List of model column names
        labels: Ground truth labels
        label: Display name for printing
        
    Returns:
        Tuple of (council_predictions, metrics_dict)
    """
    # Apply majority vote row-by-row
    council_preds = table[model_cols].apply(
        lambda row: majority_vote(row.tolist()), axis=1
    ).tolist()
    
    print(f"\n{'─'*60}")
    m = print_metrics(label, labels, council_preds)
    return council_preds, m


def cohen_kappa_pair(preds_a, preds_b, name_a, name_b):
    """
    Compute Cohen's kappa for a pair of raters (excludes invalid predictions).
    
    Cohen's kappa measures agreement corrected for chance.
    Values: <0 (worse than chance), 0 (chance), 1 (perfect agreement)
    
    Args:
        preds_a: Predictions from first rater
        preds_b: Predictions from second rater
        name_a: Display name for first rater
        name_b: Display name for second rater
        
    Returns:
        Cohen's kappa value or None if no valid pairs
    """
    # Filter to rows where both raters have valid predictions
    pairs = [(a, b) for a, b in zip(preds_a, preds_b) if a != -1 and b != -1]
    if not pairs:
        print(f"  No valid pairs for {name_a} vs {name_b}")
        return None
    a_vals, b_vals = zip(*pairs)
    k = cohen_kappa_score(a_vals, b_vals)
    print(f"  Cohen's κ  {name_a} vs {name_b} : {k:.4f}")
    return k


def multi_rater_fleiss(table, model_cols, label=''):
    """
    Compute Fleiss' kappa for multi-rater agreement.
    
    Fleiss' kappa extends Cohen's kappa to multiple raters.
    Excludes rows with any invalid predictions.
    
    Args:
        table: DataFrame with model predictions as columns
        model_cols: List of model column names
        label: Display label for printing
        
    Returns:
        Fleiss' kappa value or None if no complete rows
    """
    sub = table[model_cols].copy()
    # Require all models to have valid predictions
    valid_rows = sub[(sub != -1).all(axis=1)]
    if valid_rows.empty:
        print(f"  No complete rows for Fleiss κ ({label})")
        return None
    
    # Convert to rater count format for Fleiss' kappa
    agg, _ = aggregate_raters(valid_rows.values)
    k = fleiss_kappa(agg)
    print(f"  Fleiss' κ  ({label}, n={len(valid_rows)}) : {k:.4f}")
    return k

In [ ]:
# Apply council approach (majority voting) to Phase 2 classification table.
# Computes Cohen's kappa between reasoning models and Fleiss' kappa across all models.

# ─────────────────────────────────────────────────────────────────────────────
# Load Phase 2 classification table
# ─────────────────────────────────────────────────────────────────────────────
step2_table  = pd.read_csv(RESULTS_DIR / 'step2_classifications.csv')
model_cols   = [m[1] for m in ALL_MODELS]
reasoning_cols = [s for s in model_cols if s in reasoning_save_names]

print("STEP 4 - Council on STEP 2 table")

# ─────────────────────────────────────────────────────────────────────────────
# Apply majority vote and compute metrics
# ─────────────────────────────────────────────────────────────────────────────
s2_council_preds, s2_council_m = council_metrics(
    step2_table, model_cols, labels_full, label='Step2 majority vote'
)

# ─────────────────────────────────────────────────────────────────────────────
# Compute Cohen's kappa between reasoning models (GPT vs Gemini)
# ─────────────────────────────────────────────────────────────────────────────
if len(reasoning_cols) == 2:
    ra, rb = reasoning_cols
    s2_kappa_cohen = cohen_kappa_pair(
        step2_table[ra].tolist(),
        step2_table[rb].tolist(),
        ra, rb
    )

# ─────────────────────────────────────────────────────────────────────────────
# Compute Fleiss' kappa across all 7 models
# ─────────────────────────────────────────────────────────────────────────────
s2_kappa_fleiss = multi_rater_fleiss(
    step2_table, model_cols, label='Step2 all models'
)

# Store results for summary
s2_council_row = {
    'step': 'step2', 'approach': 'majority_vote',
    'cohen_kappa_reasoning': s2_kappa_cohen,
    'fleiss_kappa_all': s2_kappa_fleiss,
}
s2_council_row.update(s2_council_m)

In [ ]:
# Apply council approach (majority voting) to Phase 3 classification table.
# Computes inter-rater agreement metrics for CoT/Reasoning predictions.

# ─────────────────────────────────────────────────────────────────────────────
# Load Phase 3 classification table
# ─────────────────────────────────────────────────────────────────────────────
step3_table = pd.read_csv(RESULTS_DIR / 'step3_classifications.csv')

print("STEP 4: Council on STEP 3 table")

# ─────────────────────────────────────────────────────────────────────────────
# Apply majority vote and compute metrics
# ─────────────────────────────────────────────────────────────────────────────
s3_council_preds, s3_council_m = council_metrics(
    step3_table, model_cols, labels_full, label='Step3 majority vote'
)

# ─────────────────────────────────────────────────────────────────────────────
# Compute Cohen's kappa between reasoning models
# ─────────────────────────────────────────────────────────────────────────────
if len(reasoning_cols) == 2:
    ra, rb = reasoning_cols
    s3_kappa_cohen = cohen_kappa_pair(
        step3_table[ra].tolist(),
        step3_table[rb].tolist(),
        ra, rb
    )

# ─────────────────────────────────────────────────────────────────────────────
# Compute Fleiss' kappa across all 7 models
# ─────────────────────────────────────────────────────────────────────────────
s3_kappa_fleiss = multi_rater_fleiss(
    step3_table, model_cols, label='Step3 all models'
)

# Store results for summary
s3_council_row = {
    'step': 'step3', 'approach': 'majority_vote',
    'cohen_kappa_reasoning': s3_kappa_cohen,
    'fleiss_kappa_all': s3_kappa_fleiss,
}
s3_council_row.update(s3_council_m)

In [ ]:
# Consolidate and save Phase 4 council approach summary comparing Phase 2 and Phase 3 results.
# Shows whether reasoning/CoT improved model agreement and classification performance.

# ─────────────────────────────────────────────────────────────────────────────
# Combine Phase 2 and Phase 3 council results
# ─────────────────────────────────────────────────────────────────────────────
council_summary = pd.DataFrame([s2_council_row, s3_council_row])
council_summary.to_csv(RESULTS_DIR / 'step4_council_summary.csv', index=False)

# ─────────────────────────────────────────────────────────────────────────────
# Display final summary
# ─────────────────────────────────────────────────────────────────────────────
print("STEP 4 SUMMARY")
print(council_summary[['step','f2','rec','prec','acc',
                        'cohen_kappa_reasoning','fleiss_kappa_all']].to_string(index=False))
print(f"\n✓ Saved: step4_council_summary.csv")